**PSO: Particle Swarm Optimization**

A mathematical modelling algorithm, where the goal is to find the global minima of a fitness function.

**PSO with Classification Models**

To ensure a fair comparison against Encoders with genetic algorithm, the PSO will be trained on NBC model, then run by SVM and RF models.

In [5]:
import pyswarms as ps
from pyswarms.discrete import BinaryPSO
from pyswarms.utils.search import RandomSearch
from pyswarms.single import GlobalBestPSO
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

X_df = pd.read_excel('minmax.xlsx')
y_df = pd.read_csv('idC_with_header.csv')

X = X_df.to_numpy()
y = y_df.to_numpy()

options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pso = BinaryPSO(n_particles=30, dimensions=X.shape[1], options=options)

import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def fitness(particles):
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)
    alpha = 0.99
    for i, particle in enumerate(particles):
        mask = particle.astype(bool)
        if not mask.any():
            scores[i] = 1.0
            continue
        X_sub = X[:, mask]
        acc = cross_val_score(GaussianNB(var_smoothing=1e-9),
                              X_sub, y,
                              cv=5,
                              scoring='accuracy',
                              n_jobs=-1).mean()
        sparsity = mask.sum() / X.shape[1]
        scores[i] = alpha * (1 - acc) + (1 - alpha) * sparsity
    return scores

best_cost, best_pos = pso.optimize(fitness, iters=100)
print("Best Cost: ", best_cost, "\nBest POS: ", best_pos)

selected = np.where(best_pos == 1)[0]
X_selected = X[:, selected]

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_preds)
print(f"SVM Accuracy on features selected by PSO: {svm_acc * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(X_train, y_train)
nbc_preds = nbc_model.predict(X_test)
nbc_acc = accuracy_score(y_test, nbc_preds)
print(f"Naïve Bayes Accuracy on features selected by PSO: {nbc_acc * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)
print(f"Random Forest Accuracy on features selected by PSO: {rf_acc * 100:.4f}")

2025-04-19 17:12:31,732 - pyswarms.discrete.binary - INFO - Optimize for 100 iters with {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pyswarms.discrete.binary: 100%|██████████|100/100, best_cost=0.0917
2025-04-19 17:20:06,958 - pyswarms.discrete.binary - INFO - Optimization finished | best cost: 0.09168464419475662, best pos: [0 0 0 1 0 1 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 1 0 0 1 1 0 0 0 1 1 1 0 0 1
 0 0 0 0 1 1 0 1 0 0 1 1 0 0 0 0 1 1 0 0 1 0 1 0 1 0 1 0 0 1 1 1 0 0 1 1 0
 1 1 0 1 0 0 0 1 0 1 0 0 1 0 0 0 0 0 0 0 1 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 1
 0 1 1 1 0 0 1 1 1 1 1 1 0 0 0 1 0 1 0 0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 0 0 1
 1 0 0 1 1 0 1 0 0 0 0 1 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 1 1 1 0
 1 1 1 0 1 1 0 1 0 0 1 1 1 0 1 0 1 0 1 1 0 1 0 0 0 0 1 0 1 0 1 0 1 1 0 1 0
 1 1 0 1 0 0 0 0 1 0 0 0 1 1 0 1 1 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 1 1
 1 1 1 1 1 0 1 1 1 0 0 1 0 0 1 0 1 0 0 1 1 0 0 1 1 1 0 1 0 0 1 1 1 0 1 0 1
 1 1 0 0 0 0 1 0 1 0 1 0 1 0 1 1 0 1 1 0 1 1 0 1 0 0 1 1 1 1 0 0 0 0 1 1 

Best Cost:  0.09168464419475662 
Best POS:  [0 0 0 1 0 1 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 1 0 0 1 1 0 0 0 1 1 1 0 0 1
 0 0 0 0 1 1 0 1 0 0 1 1 0 0 0 0 1 1 0 0 1 0 1 0 1 0 1 0 0 1 1 1 0 0 1 1 0
 1 1 0 1 0 0 0 1 0 1 0 0 1 0 0 0 0 0 0 0 1 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 1
 0 1 1 1 0 0 1 1 1 1 1 1 0 0 0 1 0 1 0 0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 0 0 1
 1 0 0 1 1 0 1 0 0 0 0 1 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 1 1 1 0
 1 1 1 0 1 1 0 1 0 0 1 1 1 0 1 0 1 0 1 1 0 1 0 0 0 0 1 0 1 0 1 0 1 1 0 1 0
 1 1 0 1 0 0 0 0 1 0 0 0 1 1 0 1 1 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 1 1
 1 1 1 1 1 0 1 1 1 0 0 1 0 0 1 0 1 0 0 1 1 0 0 1 1 1 0 1 0 0 1 1 1 0 1 0 1
 1 1 0 0 0 0 1 0 1 0 1 0 1 0 1 1 0 1 1 0 1 1 0 1 0 0 1 1 1 1 0 0 0 0 1 1 0
 1 0 0 0 0 0 0 1 0 1 0 0 1 1 0 1 1 0 0 0 0 0 1 0 1 1 0 0 0 0 1 1 1 0 1 1 1
 0 1 0 0 0 1 1 0 1 0 0 1 0 1 1 1 0 0 0 1 0 0 1 1 0 1 1 0 1 1 0 0 0 0 1 0 1
 0 0 1 0 1 0 1 1 0 0 1 0 1 1 0 1 0 0 0 0 1 0 0 1 0 0 1 0 1 1 0 1 0 0 0 0 0
 1 1 0 1 0 0 1 1 1 0 1 0 1 0 1 0 0 0 1 0 0 0 1 1 0 1 0 0

**Summary**

Accuracy results for models is slightly less than that of Encoders with GA implementation.

- SVM with POS: 95%
- NBC with POS: 87%
- RF with POS: 93%